In [8]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers import MultiQueryRetriever

In [9]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [14]:
from dotenv import load_dotenv
import os
load_dotenv()

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENAI_API_KEY")
)

In [15]:
vectorstore = FAISS.from_documents(
    documents=all_docs,
    embedding=embeddings
)

In [16]:
similarity_retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k':5}
)

In [26]:
multiquery_retriver = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs = {'k':5}),
    llm=ChatOpenAI(
        model="gpt-4o-mini",
        base_url="https://openrouter.ai/api/v1",
    )
)

In [27]:
# Query
query = "How to improve energy levels and maintain balance?"

In [28]:
similarity_result = similarity_retriever.invoke(query)
multiquery_result = multiquery_retriver.invoke(query)

In [29]:
for i, doc in enumerate(similarity_result):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_result):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 5 ---
Deep sleep is crucial for cellular repair and emotional regulation.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
Deep sleep is crucial for cellular repair and emotional regulation

In [33]:
for i, doc in enumerate(multiquery_result):
    print(f"--------{i+1}")
    print(doc.page_content)

--------1
Drinking sufficient water throughout the day helps maintain metabolism and energy.
--------2
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
--------3
Regular walking boosts heart health and can reduce symptoms of depression.
--------4
Deep sleep is crucial for cellular repair and emotional regulation.
--------5
Consuming leafy greens and fruits helps detox the body and improve longevity.
--------6
The solar energy system in modern homes helps balance electricity demand.
